# Phase 0 — Auto Ball Tracking Validation

Run order: top to bottom. Use **Runtime → Change runtime type → T4 GPU** before starting.

Outputs `tracks.json` + `preview_tv.mp4` + `preview_compare.mp4`. Watch the preview videos to decide go/no-go on the full pipeline.

## 1. Install dependencies

In [1]:
# If running on the local Mac venv (~/.venvs/match-tracker-phase0), deps are already installed — this cell is a no-op.
# To re-install via corporate artifactory (Mac/Apple Silicon, Python 3.12), uncomment:
# %pip install -q --index-url https://artifactory.foc.zone/artifactory/api/pypi/rdf-pypi-virtual/simple --extra-index-url https://artifactory.foc.zone/artifactory/api/pypi/pypi-remote/simple "torch>=2.2,<2.6" "torchvision<0.21" opencv-python-headless "numpy<2" tqdm pillow pyyaml requests
# %pip install -q --no-deps --index-url https://artifactory.foc.zone/artifactory/api/pypi/rdf-pypi-virtual/simple --extra-index-url https://artifactory.foc.zone/artifactory/api/pypi/pypi-remote/simple ultralytics ultralytics-thop py-cpuinfo

import torch, platform

if torch.cuda.is_available():
    DEVICE = 'cuda'
    name = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    name = f'Apple Silicon MPS ({platform.machine()})'
else:
    DEVICE = 'cpu'
    name = 'CPU'
print(f'Device: {DEVICE} — {name}')


Device: mps — Apple Silicon MPS (arm64)


## 2. Provide input video

Either upload a clip from your Insta360 X5 (stitched equirectangular MP4), or set `INPUT_URL` to a sample. Keep clips ≤ 60 sec for fast iteration.

In [3]:
import os, cv2

# Practice recording from May 28, 2026 — stitched equirectangular export.
INPUT_PATH = '/Users/irezaeian/match-tracker/tracking/samples/VID_test.mp4'
assert os.path.isfile(INPUT_PATH), f'Sample not found at {INPUT_PATH}'

OUT_DIR = '/Users/irezaeian/match-tracker/tracking/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

cap = cv2.VideoCapture(INPUT_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# True 2:1 equirectangular vs YouTube-repacked 16:9 (or any other aspect).
IS_EQUIRECTANGULAR = abs(W / H - 2.0) < 0.05

print(f'Input:  {INPUT_PATH}')
print(f'Format: {W}x{H} @ {FPS:.1f}fps, {N} frames, {N/FPS:.1f}s')
print(f'Output: {OUT_DIR}')
print(f'Mode:   {"equirectangular (true 2:1)" if IS_EQUIRECTANGULAR else "flat (non-2:1; perspective render will use crop/pan)"}')

Input:  /Users/irezaeian/match-tracker/tracking/samples/VID_test.mp4
Format: 5760x2880 @ 30.0fps, 1468 frames, 49.0s
Output: /Users/irezaeian/match-tracker/tracking/outputs
Mode:   equirectangular (true 2:1)


## 3. Load YOLOv8 detector

Using `yolov8n.pt` (smallest, fastest). COCO-pretrained → classes 0=person, 32=sports ball. Good enough for validation; we fine-tune later if needed.

In [4]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.to(DEVICE)
print(f'Model loaded on {DEVICE}.')

Model loaded on mps.


## 4. Run detection per frame

**Strategy for equirectangular**: action stays near the equator (mid 33% vertically) on a level-tripod 360° rig. We crop that band before YOLO to (a) reduce distortion and (b) speed up inference 3x.

Coordinate output: `lon ∈ [-180, 180]`, `lat ∈ [-90, 90]`. Lon=0 is the direction the camera was facing at start.

In [5]:
from tqdm import tqdm
import numpy as np

# Process at 10 Hz (every ~3 frames at 30fps) — matches output sample rate
SAMPLE_HZ = 10
stride = max(1, round(FPS / SAMPLE_HZ))

# Equator band crop (middle 33% vertically) — only when truly equirectangular.
# For flat (16:9) input the field already fills the frame, so use the full frame.
if IS_EQUIRECTANGULAR:
    BAND_TOP = int(H * 0.33)
    BAND_BOT = int(H * 0.67)
else:
    BAND_TOP = 0
    BAND_BOT = H
BAND_H = BAND_BOT - BAND_TOP

def px_to_lonlat(cx, cy):
    # For 2:1 equirectangular: full sphere mapping.
    # For flat input: scale x → [-180, 180] across the frame, y → [-90, 90].
    # This keeps the same downstream code (smoothing in degrees) without crashing.
    lon = (cx / W) * 360.0 - 180.0
    lat = 90.0 - (cy / H) * 180.0
    return lon, lat

cap = cv2.VideoCapture(INPUT_PATH)
raw_detections = []  # list of dicts per processed frame

frame_idx = 0
pbar = tqdm(total=N, desc='Detecting')
while True:
    ret, frame = cap.read()
    if not ret: break
    if frame_idx % stride == 0:
        band = frame[BAND_TOP:BAND_BOT, :, :]
        # Resize for speed (keep aspect)
        det_w = 1280
        det_h = int(BAND_H * det_w / W)
        small = cv2.resize(band, (det_w, det_h))
        results = model.predict(small, conf=0.25, classes=[0, 32], verbose=False)
        players, ball = [], None
        if len(results) > 0:
            r = results[0]
            for box, cls, conf in zip(r.boxes.xyxy.cpu().numpy(),
                                       r.boxes.cls.cpu().numpy(),
                                       r.boxes.conf.cpu().numpy()):
                x1, y1, x2, y2 = box
                cx = (x1 + x2) / 2 * (W / det_w)
                cy_in_band = (y1 + y2) / 2 * (BAND_H / det_h)
                cy = BAND_TOP + cy_in_band
                lon, lat = px_to_lonlat(cx, cy)
                if int(cls) == 0:  # person
                    players.append({'lon': lon, 'lat': lat, 'conf': float(conf)})
                elif int(cls) == 32 and (ball is None or conf > ball['conf']):
                    ball = {'lon': lon, 'lat': lat, 'conf': float(conf)}
        raw_detections.append({
            't': frame_idx / FPS,
            'players': players,
            'ball': ball,
        })
    frame_idx += 1
    pbar.update(1)
pbar.close()
cap.release()
print(f'Processed {len(raw_detections)} sampled frames @ {SAMPLE_HZ}Hz')
print(f'Frames with ball detection: {sum(1 for d in raw_detections if d["ball"])} / {len(raw_detections)}')
print(f'Avg players per frame: {np.mean([len(d["players"]) for d in raw_detections]):.1f}')


Detecting: 100%|██████████| 1468/1468 [00:27<00:00, 53.50it/s]

Processed 490 sampled frames @ 10Hz
Frames with ball detection: 0 / 490
Avg players per frame: 0.6


In [6]:
# Diagnostic: detect on FULL frame (no band crop) at a few sample frames
# to see where objects actually appear in the equirectangular image.
import numpy as np

cap = cv2.VideoCapture(INPUT_PATH)
test_frames = [0, 150, 300, 450, 600, 900]  # ~0s, 5s, 10s, 15s, 20s, 30s
all_cys = []
all_cxs = []
n_persons = 0
n_balls = 0

for target_idx in test_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
    ret, frame = cap.read()
    if not ret:
        continue
    # Detect on full frame (resize to 1920 wide for speed)
    det_w = 1920
    det_h = int(H * det_w / W)
    small = cv2.resize(frame, (det_w, det_h))
    results = model.predict(small, conf=0.2, classes=[0, 32], verbose=False)
    if len(results) > 0:
        r = results[0]
        for box, cls in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy()):
            x1, y1, x2, y2 = box
            cx = (x1 + x2) / 2 / det_w  # normalized 0-1
            cy = (y1 + y2) / 2 / det_h  # normalized 0-1
            all_cxs.append(cx)
            all_cys.append(cy)
            if int(cls) == 0:
                n_persons += 1
            elif int(cls) == 32:
                n_balls += 1

cap.release()

if all_cys:
    cys = np.array(all_cys)
    print(f'Detected {n_persons} persons, {n_balls} balls across {len(test_frames)} sample frames')
    print(f'Y-position (0=top, 1=bottom):')
    print(f'  Min: {cys.min():.3f}  Max: {cys.max():.3f}  Mean: {cys.mean():.3f}  Median: {np.median(cys):.3f}')
    print(f'  Current band: 0.33 – 0.67')
    print(f'  Objects in band: {np.sum((cys >= 0.33) & (cys <= 0.67))} / {len(cys)}')
    print(f'  Objects above band (< 0.33): {np.sum(cys < 0.33)}')
    print(f'  Objects below band (> 0.67): {np.sum(cys > 0.67)}')
    # Suggest a better band
    p5, p95 = np.percentile(cys, [5, 95])
    print(f'\n  Suggested band: {p5:.2f} – {p95:.2f} (covers 90% of detections)')
else:
    print('No detections at all on full frame — the scene may not have visible players/ball.')

No detections at all on full frame — the scene may not have visible players/ball.


In [7]:
# Diagnostic: extract a frame and try detection at higher resolution + lower confidence
# Also save the frame so we can visually inspect it.
import numpy as np

cap = cv2.VideoCapture(INPUT_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 450)  # ~15 seconds in
ret, frame = cap.read()
cap.release()

print(f'Frame shape: {frame.shape}')  # Should be (2880, 5760, 3)

# Save a full-res frame for visual inspection
cv2.imwrite(os.path.join(OUT_DIR, 'debug_frame_full.jpg'), frame, [cv2.IMWRITE_JPEG_QUALITY, 85])

# Try detection at FULL 5760 width (no resize) - slow but maximizes detection
results_full = model.predict(frame, conf=0.15, classes=[0, 32], verbose=False)
print(f'\nFull-res detection (5760x2880, conf=0.15):')
if len(results_full) > 0:
    r = results_full[0]
    persons = sum(1 for c in r.boxes.cls.cpu().numpy() if int(c) == 0)
    balls = sum(1 for c in r.boxes.cls.cpu().numpy() if int(c) == 32)
    print(f'  Persons: {persons}, Balls: {balls}')
    # Print all detections with positions
    for box, cls, conf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = box
        cx, cy = (x1+x2)/2, (y1+y2)/2
        label = 'person' if int(cls) == 0 else 'ball'
        w_box, h_box = x2-x1, y2-y1
        print(f'  {label}: center=({cx:.0f},{cy:.0f}) size=({w_box:.0f}x{h_box:.0f}) conf={conf:.2f} y_norm={cy/H:.3f}')
else:
    print('  No detections')

# Also try with ALL classes to see if YOLO detects anything at all
results_all = model.predict(frame, conf=0.15, verbose=False)
print(f'\nAll-class detection (any object, conf=0.15):')
if len(results_all) > 0:
    r = results_all[0]
    print(f'  Total detections: {len(r.boxes)}')
    from collections import Counter
    class_counts = Counter(int(c) for c in r.boxes.cls.cpu().numpy())
    class_names = r.names
    for cls_id, count in class_counts.most_common(10):
        print(f'  {class_names[cls_id]}: {count}')
else:
    print('  No detections at all')

Frame shape: (2880, 5760, 3)

Full-res detection (5760x2880, conf=0.15):
  Persons: 0, Balls: 0

All-class detection (any object, conf=0.15):
  Total detections: 0


In [8]:
# Deeper diagnostic: check if video is readable & try CPU inference on a crop
import numpy as np

cap = cv2.VideoCapture(INPUT_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 450)
ret, frame = cap.read()
cap.release()

print(f'Frame shape: {frame.shape}, dtype: {frame.dtype}')
print(f'Pixel values — min: {frame.min()}, max: {frame.max()}, mean: {frame.mean():.1f}')

# Is the frame all black/corrupted?
if frame.mean() < 5:
    print('WARNING: Frame appears nearly black — video may not be readable at this position')

# Save 4 quadrant crops for visual inspection
h, w = frame.shape[:2]
quadrants = {
    'top_left': frame[:h//2, :w//4],
    'top_center': frame[:h//2, w//4:3*w//4],
    'top_right': frame[:h//2, 3*w//4:],
    'bottom_center': frame[h//2:, w//4:3*w//4],
}
for name, crop in quadrants.items():
    path = os.path.join(OUT_DIR, f'debug_{name}.jpg')
    cv2.imwrite(path, crop, [cv2.IMWRITE_JPEG_QUALITY, 80])
print(f'\nSaved quadrant crops to {OUT_DIR}/debug_*.jpg')

# Try detection on CPU with a manageable crop (center 1920x960)
cx_start, cy_start = w//2 - 960, h//2 - 480
center_crop = frame[cy_start:cy_start+960, cx_start:cx_start+1920]
print(f'\nCenter crop shape: {center_crop.shape}')
cv2.imwrite(os.path.join(OUT_DIR, 'debug_center_crop.jpg'), center_crop, [cv2.IMWRITE_JPEG_QUALITY, 85])

# Force CPU inference to rule out MPS issues
results_cpu = model.predict(center_crop, conf=0.1, device='cpu', verbose=False)
print(f'CPU inference on center crop (conf=0.1):')
if len(results_cpu) > 0 and len(results_cpu[0].boxes) > 0:
    r = results_cpu[0]
    print(f'  Detections: {len(r.boxes)}')
    for box, cls, conf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        print(f'  {r.names[int(cls)]}: conf={conf:.2f}')
else:
    print('  No detections on center crop either')

# Try on multiple different frames to check if video is OK
cap = cv2.VideoCapture(INPUT_PATH)
print(f'\nSpot-check pixel means at different timestamps:')
for fnum in [0, 100, 300, 600, 900, 1200]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, fnum)
    ret, f = cap.read()
    if ret:
        print(f'  Frame {fnum} ({fnum/30:.1f}s): mean={f.mean():.1f}, shape={f.shape}')
    else:
        print(f'  Frame {fnum}: READ FAILED')
cap.release()

Frame shape: (2880, 5760, 3), dtype: uint8
Pixel values — min: 0, max: 255, mean: 131.8

Saved quadrant crops to /Users/irezaeian/match-tracker/tracking/outputs/debug_*.jpg

Center crop shape: (960, 1920, 3)
CPU inference on center crop (conf=0.1):
  Detections: 15
  person: conf=0.70
  person: conf=0.64
  person: conf=0.55
  person: conf=0.50
  person: conf=0.49
  person: conf=0.45
  person: conf=0.44
  person: conf=0.35
  person: conf=0.24
  person: conf=0.23
  person: conf=0.18
  person: conf=0.18
  person: conf=0.15
  person: conf=0.11
  person: conf=0.11

Spot-check pixel means at different timestamps:
  Frame 0 (0.0s): mean=132.1, shape=(2880, 5760, 3)
  Frame 100 (3.3s): mean=132.2, shape=(2880, 5760, 3)
  Frame 300 (10.0s): mean=131.8, shape=(2880, 5760, 3)
  Frame 600 (20.0s): mean=131.7, shape=(2880, 5760, 3)
  Frame 900 (30.0s): mean=132.5, shape=(2880, 5760, 3)
  Frame 1200 (40.0s): mean=132.1, shape=(2880, 5760, 3)


In [9]:
# Re-run detection with fixes:
# 1. Use CPU (MPS fails silently on large equirectangular frames)
# 2. Use center crop (drill is around one goal, not full 360)
# 3. Lower confidence for ball (it's small in a drill)
from tqdm import tqdm
import numpy as np

# For this drill, the action is concentrated in one area. 
# We'll scan the center 180° (middle half of the equirectangular width)
# and the equator band (middle 50% height — wider than before).
CROP_X_START = W // 4        # start at 25% of width
CROP_X_END = 3 * W // 4     # end at 75% of width
CROP_Y_START = int(H * 0.25)  # top 25%
CROP_Y_END = int(H * 0.75)    # bottom 75%
CROP_W = CROP_X_END - CROP_X_START  # 2880
CROP_H = CROP_Y_END - CROP_Y_START  # 1440

# Detection resolution — keep crop at 1920 wide for speed
DET_W = 1920
DET_H = int(CROP_H * DET_W / CROP_W)

SAMPLE_HZ = 10
stride = max(1, round(FPS / SAMPLE_HZ))

cap = cv2.VideoCapture(INPUT_PATH)
raw_detections_v2 = []

frame_idx = 0
pbar = tqdm(total=N, desc='Detecting (CPU, center crop)')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % stride == 0:
        crop = frame[CROP_Y_START:CROP_Y_END, CROP_X_START:CROP_X_END]
        small = cv2.resize(crop, (DET_W, DET_H))
        results = model.predict(small, conf=0.15, classes=[0, 32], device='cpu', verbose=False)
        players, ball = [], None
        if len(results) > 0:
            r = results[0]
            for box, cls, conf in zip(r.boxes.xyxy.cpu().numpy(),
                                       r.boxes.cls.cpu().numpy(),
                                       r.boxes.conf.cpu().numpy()):
                x1, y1, x2, y2 = box
                # Map back to full-frame coordinates
                cx = CROP_X_START + (x1 + x2) / 2 * (CROP_W / DET_W)
                cy = CROP_Y_START + (y1 + y2) / 2 * (CROP_H / DET_H)
                lon = (cx / W) * 360.0 - 180.0
                lat = 90.0 - (cy / H) * 180.0
                if int(cls) == 0:
                    players.append({'lon': lon, 'lat': lat, 'conf': float(conf)})
                elif int(cls) == 32 and (ball is None or conf > ball['conf']):
                    ball = {'lon': lon, 'lat': lat, 'conf': float(conf)}
        raw_detections_v2.append({
            't': frame_idx / FPS,
            'players': players,
            'ball': ball,
        })
    frame_idx += 1
    pbar.update(1)
pbar.close()
cap.release()

ball_frames = sum(1 for d in raw_detections_v2 if d['ball'])
print(f'\nProcessed {len(raw_detections_v2)} frames @ {SAMPLE_HZ}Hz')
print(f'Frames with ball: {ball_frames} / {len(raw_detections_v2)} ({100*ball_frames/len(raw_detections_v2):.1f}%)')
print(f'Avg players/frame: {np.mean([len(d["players"]) for d in raw_detections_v2]):.1f}')
print(f'Max players/frame: {max(len(d["players"]) for d in raw_detections_v2)}')

Detecting (CPU, center crop): 100%|██████████| 1468/1468 [00:30<00:00, 48.88it/s]


Processed 490 frames @ 10Hz
Frames with ball: 1 / 490 (0.2%)
Avg players/frame: 7.6
Max players/frame: 18


In [13]:
# Smoothing + TV-mode render using PLAYER CENTROID only (no ball).
# For a shooting drill the "action" is where the cluster of players is.
import numpy as np

EMA_ALPHA = 0.04  # very smooth panning (was 0.12 — too jittery)
DEAD_ZONE_DEG = 4.0  # don't move camera if target shifts < 4°

# Compute player centroid per frame (weighted by confidence)
track_points = []
for det in raw_detections_v2:
    if det['players']:
        lons = np.array([p['lon'] for p in det['players']])
        lats = np.array([p['lat'] for p in det['players']])
        confs = np.array([p['conf'] for p in det['players']])
        confs = confs / confs.sum()  # normalize to weights
        lon_c = float(np.average(lons, weights=confs))
        lat_c = float(np.average(lats, weights=confs))
        track_points.append({'t': det['t'], 'lon': lon_c, 'lat': lat_c})
    elif track_points:
        # Hold last position
        track_points.append({'t': det['t'], 'lon': track_points[-1]['lon'], 'lat': track_points[-1]['lat']})
    else:
        track_points.append({'t': det['t'], 'lon': 0.0, 'lat': 0.0})

# EMA smooth with dead zone
smoothed = []
cam_lon, cam_lat = track_points[0]['lon'], track_points[0]['lat']
for pt in track_points:
    d_lon = pt['lon'] - cam_lon
    d_lat = pt['lat'] - cam_lat
    dist = (d_lon**2 + d_lat**2)**0.5
    if dist > DEAD_ZONE_DEG:
        cam_lon += EMA_ALPHA * d_lon
        cam_lat += EMA_ALPHA * d_lat
    smoothed.append({'t': pt['t'], 'lon': cam_lon, 'lat': cam_lat})

# Second pass: Gaussian-like smoothing (simple moving average over smoothed track)
WINDOW = 15  # 15 samples at 10Hz = 1.5s averaging window
smoothed2 = []
for i in range(len(smoothed)):
    lo = max(0, i - WINDOW // 2)
    hi = min(len(smoothed), i + WINDOW // 2 + 1)
    avg_lon = sum(s['lon'] for s in smoothed[lo:hi]) / (hi - lo)
    avg_lat = sum(s['lat'] for s in smoothed[lo:hi]) / (hi - lo)
    smoothed2.append({'t': smoothed[i]['t'], 'lon': avg_lon, 'lat': avg_lat})
smoothed = smoothed2

print(f'Track points: {len(smoothed)}')
print(f'Camera range: lon [{min(s["lon"] for s in smoothed):.1f}, {max(s["lon"] for s in smoothed):.1f}]')
print(f'              lat [{min(s["lat"] for s in smoothed):.1f}, {max(s["lat"] for s in smoothed):.1f}]')

Track points: 490
Camera range: lon [-31.0, 1.3]
              lat [-19.3, -14.5]


In [14]:
# Render TV-mode preview: re-project equirectangular through a virtual perspective camera
# following the smoothed track.
import math

OUT_W, OUT_H = 1280, 720  # 16:9 TV crop
FOV_DEG = 70  # field of view (wider = more context, narrower = more zoom)

def render_perspective(equi, lon_deg, lat_deg, fov_deg, out_w, out_h):
    """Sample equirectangular frame through a perspective virtual camera."""
    h_eq, w_eq = equi.shape[:2]
    f = out_w / (2 * math.tan(math.radians(fov_deg) / 2))

    # Pixel grid for output
    u = np.arange(out_w) - out_w / 2
    v = np.arange(out_h) - out_h / 2
    uu, vv = np.meshgrid(u, v)

    # Ray directions in camera space (z = forward, y = up, x = right)
    x = uu
    y = -vv
    z = np.full_like(uu, f, dtype=np.float64)
    norm = np.sqrt(x**2 + y**2 + z**2)
    x, y, z = x/norm, y/norm, z/norm

    # Rotate by lon/lat
    lon_r = math.radians(lon_deg)
    lat_r = math.radians(-lat_deg)  # negate: positive lat = look up in equirect, but our lat is "up from equator"

    # Rotation: first pitch (lat around X), then yaw (lon around Y)
    cos_lat, sin_lat = math.cos(lat_r), math.sin(lat_r)
    cos_lon, sin_lon = math.cos(lon_r), math.sin(lon_r)

    # Pitch (around X axis)
    y2 = y * cos_lat - z * sin_lat
    z2 = y * sin_lat + z * cos_lat

    # Yaw (around Y axis)
    x3 = x * cos_lon + z2 * sin_lon
    z3 = -x * sin_lon + z2 * cos_lon
    y3 = y2

    # Convert to equirectangular coordinates
    theta = np.arctan2(x3, z3)  # longitude [-pi, pi]
    phi = np.arcsin(np.clip(y3, -1, 1))  # latitude [-pi/2, pi/2]

    # Map to pixel coordinates in equirectangular image
    map_x = ((theta / math.pi + 1) / 2 * w_eq).astype(np.float32)
    map_y = ((0.5 - phi / math.pi) * h_eq).astype(np.float32)

    # Clamp
    map_x = np.clip(map_x, 0, w_eq - 1)
    map_y = np.clip(map_y, 0, h_eq - 1)

    return cv2.remap(equi, map_x, map_y, cv2.INTER_LINEAR)

# Build lookup: frame_idx → smoothed camera position
# smoothed is at 10Hz, video is at 30fps — use LINEAR INTERPOLATION to avoid step-jumps
import bisect
smooth_times = [s['t'] for s in smoothed]

def get_cam_at(t):
    """Linearly interpolate between adjacent smoothed samples for buttery panning."""
    idx = bisect.bisect_right(smooth_times, t) - 1
    idx = max(0, min(idx, len(smoothed) - 2))
    # Interpolate between idx and idx+1
    t0 = smoothed[idx]['t']
    t1 = smoothed[idx + 1]['t']
    dt = t1 - t0
    if dt < 1e-6:
        frac = 0.0
    else:
        frac = (t - t0) / dt
        frac = max(0.0, min(1.0, frac))
    lon = smoothed[idx]['lon'] + frac * (smoothed[idx + 1]['lon'] - smoothed[idx]['lon'])
    lat = smoothed[idx]['lat'] + frac * (smoothed[idx + 1]['lat'] - smoothed[idx]['lat'])
    return lon, lat

# Render
out_path = os.path.join(OUT_DIR, 'preview_tv_smooth.mp4')
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(out_path, fourcc, FPS, (OUT_W, OUT_H))

cap = cv2.VideoCapture(INPUT_PATH)
frame_idx = 0
pbar = tqdm(total=N, desc='Rendering TV preview (smooth)')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    t = frame_idx / FPS
    lon, lat = get_cam_at(t)
    tv_frame = render_perspective(frame, lon, lat, FOV_DEG, OUT_W, OUT_H)
    writer.write(tv_frame)
    frame_idx += 1
    pbar.update(1)
pbar.close()
cap.release()
writer.release()

file_size = os.path.getsize(out_path) / 1024 / 1024
print(f'\nRendered: {out_path}')
print(f'Size: {file_size:.1f} MB, {frame_idx} frames @ {FPS:.1f}fps')

Rendering TV preview (smooth): 100%|██████████| 1468/1468 [00:57<00:00, 25.37it/s]


Rendered: /Users/irezaeian/match-tracker/tracking/outputs/preview_tv_smooth.mp4
Size: 68.2 MB, 1468 frames @ 30.0fps


## Hybrid approach: Player centroid → crop → ball search

Use the smoothed player centroid as coarse aim, extract the virtual viewport,
then run ball detection on that smaller crop where the ball is 3-4x larger in pixels.
If ball is found, blend it into the camera target (ball attracts camera toward action).

In [15]:
# Pass 2: Ball search within the virtual viewport
# Strategy: for each sampled frame, extract perspective crop at the smoothed player centroid,
# run YOLO ball detection on that crop. If found, compute ball's lon/lat in equirectangular space.

BALL_CLASS = 32  # COCO class for 'sports ball'
BALL_CONF = 0.15  # low threshold — we want to catch faint detections
BALL_SEARCH_FOV = 80  # slightly wider than TV FOV to catch balls at edges
BALL_WEIGHT = 0.4  # blend: 60% player centroid, 40% ball position

# We'll search at the same 10Hz sample rate as player detection
ball_detections = []  # list of {'t': float, 'lon': float, 'lat': float, 'conf': float} or None

cap = cv2.VideoCapture(INPUT_PATH)
stride = int(FPS / SAMPLE_HZ)
frame_idx = 0
ball_found_count = 0

pbar = tqdm(total=N, desc='Pass 2: Ball search in viewport')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % stride == 0:
        t = frame_idx / FPS
        # Get coarse aim from smoothed player centroid
        lon_aim, lat_aim = get_cam_at(t)
        
        # Extract perspective crop at that aim point
        crop_for_ball = render_perspective(frame, lon_aim, lat_aim, BALL_SEARCH_FOV, OUT_W, OUT_H)
        
        # Run YOLO on the crop — only looking for ball
        results_ball = model.predict(crop_for_ball, conf=BALL_CONF, classes=[BALL_CLASS],
                                     device=DEVICE, verbose=False)
        
        ball_det = None
        if len(results_ball[0].boxes) > 0:
            # Take highest confidence ball
            boxes = results_ball[0].boxes
            best_idx = boxes.conf.argmax()
            bx1, by1, bx2, by2 = boxes.xyxy[best_idx].cpu().numpy()
            ball_cx = (bx1 + bx2) / 2
            ball_cy = (by1 + by2) / 2
            ball_conf = float(boxes.conf[best_idx])
            
            # Convert pixel position in crop back to lon/lat
            # The crop is centered at (lon_aim, lat_aim) with BALL_SEARCH_FOV
            # Pixel (ball_cx, ball_cy) in OUT_W x OUT_H → angular offset from center
            f_len = OUT_W / (2 * math.tan(math.radians(BALL_SEARCH_FOV) / 2))
            dx_deg = math.degrees(math.atan2(ball_cx - OUT_W/2, f_len))
            dy_deg = math.degrees(math.atan2(-(ball_cy - OUT_H/2), f_len))
            
            ball_lon = lon_aim + dx_deg
            ball_lat = lat_aim + dy_deg
            ball_det = {'t': t, 'lon': ball_lon, 'lat': ball_lat, 'conf': ball_conf}
            ball_found_count += 1
        
        ball_detections.append(ball_det)
    
    frame_idx += 1
    pbar.update(1)

pbar.close()
cap.release()

total_samples = len(ball_detections)
print(f'\nBall search complete: {ball_found_count}/{total_samples} frames with ball ({100*ball_found_count/total_samples:.1f}%)')
if ball_found_count > 0:
    avg_conf = np.mean([d['conf'] for d in ball_detections if d is not None])
    print(f'Average ball confidence: {avg_conf:.3f}')

Pass 2: Ball search in viewport: 100%|██████████| 1468/1468 [00:51<00:00, 28.56it/s]


Ball search complete: 93/734 frames with ball (12.7%)
Average ball confidence: 0.293


In [16]:
# Blend ball detections into the camera track and re-render
# Where ball is found: target = (1-BALL_WEIGHT)*player_centroid + BALL_WEIGHT*ball_position
# Then re-smooth with EMA + moving average + interpolate

# Build blended track points
blended_track = []
for i, det in enumerate(ball_detections):
    base = track_points[i] if i < len(track_points) else track_points[-1]
    if det is not None:
        # Blend toward ball
        lon_blend = (1 - BALL_WEIGHT) * base['lon'] + BALL_WEIGHT * det['lon']
        lat_blend = (1 - BALL_WEIGHT) * base['lat'] + BALL_WEIGHT * det['lat']
        blended_track.append({'t': base['t'], 'lon': lon_blend, 'lat': lat_blend})
    else:
        blended_track.append(base)

# Re-smooth: EMA + dead zone
smoothed_hybrid = []
cam_lon, cam_lat = blended_track[0]['lon'], blended_track[0]['lat']
for pt in blended_track:
    d_lon = pt['lon'] - cam_lon
    d_lat = pt['lat'] - cam_lat
    dist = (d_lon**2 + d_lat**2)**0.5
    if dist > DEAD_ZONE_DEG:
        cam_lon += EMA_ALPHA * d_lon
        cam_lat += EMA_ALPHA * d_lat
    smoothed_hybrid.append({'t': pt['t'], 'lon': cam_lon, 'lat': cam_lat})

# Second pass: moving average
smoothed_hybrid2 = []
for i in range(len(smoothed_hybrid)):
    lo = max(0, i - WINDOW // 2)
    hi = min(len(smoothed_hybrid), i + WINDOW // 2 + 1)
    avg_lon = sum(s['lon'] for s in smoothed_hybrid[lo:hi]) / (hi - lo)
    avg_lat = sum(s['lat'] for s in smoothed_hybrid[lo:hi]) / (hi - lo)
    smoothed_hybrid2.append({'t': smoothed_hybrid[i]['t'], 'lon': avg_lon, 'lat': avg_lat})
smoothed_hybrid = smoothed_hybrid2

# Update get_cam_at to use hybrid track
smooth_times_h = [s['t'] for s in smoothed_hybrid]

def get_cam_at_hybrid(t):
    """Linearly interpolate between adjacent hybrid smoothed samples."""
    idx = bisect.bisect_right(smooth_times_h, t) - 1
    idx = max(0, min(idx, len(smoothed_hybrid) - 2))
    t0 = smoothed_hybrid[idx]['t']
    t1 = smoothed_hybrid[idx + 1]['t']
    dt = t1 - t0
    if dt < 1e-6:
        frac = 0.0
    else:
        frac = max(0.0, min(1.0, (t - t0) / dt))
    lon = smoothed_hybrid[idx]['lon'] + frac * (smoothed_hybrid[idx + 1]['lon'] - smoothed_hybrid[idx]['lon'])
    lat = smoothed_hybrid[idx]['lat'] + frac * (smoothed_hybrid[idx + 1]['lat'] - smoothed_hybrid[idx]['lat'])
    return lon, lat

# Render hybrid version
out_path_hybrid = os.path.join(OUT_DIR, 'preview_tv_hybrid.mp4')
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(out_path_hybrid, fourcc, FPS, (OUT_W, OUT_H))

cap = cv2.VideoCapture(INPUT_PATH)
frame_idx = 0
pbar = tqdm(total=N, desc='Rendering hybrid TV preview')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    t = frame_idx / FPS
    lon, lat = get_cam_at_hybrid(t)
    tv_frame = render_perspective(frame, lon, lat, FOV_DEG, OUT_W, OUT_H)
    writer.write(tv_frame)
    frame_idx += 1
    pbar.update(1)
pbar.close()
cap.release()
writer.release()

file_size = os.path.getsize(out_path_hybrid) / 1024 / 1024
print(f'\nHybrid rendered: {out_path_hybrid}')
print(f'Size: {file_size:.1f} MB, {frame_idx} frames @ {FPS:.1f}fps')
print(f'Ball influenced {ball_found_count} of {total_samples} camera samples')

Rendering hybrid TV preview: 100%|██████████| 1468/1468 [01:00<00:00, 24.33it/s]


Hybrid rendered: /Users/irezaeian/match-tracker/tracking/outputs/preview_tv_hybrid.mp4
Size: 68.5 MB, 1468 frames @ 30.0fps
Ball influenced 93 of 734 camera samples


## 5. Compute action centroid + smooth

Fuse ball + player cluster into a single target point. Apply EMA + dead zone + lead. Output the camera state stream.

In [7]:
EMA_ALPHA = 0.15           # lower = smoother, higher = more responsive
BALL_WEIGHT = 0.55         # weight of ball vs player centroid (when ball detected)
LEAD_FRAMES = 6            # how many samples ahead to lead the camera
DEAD_ZONE_DEG = 8.0        # don't move if target within this of current
FOV_TIGHT = 35.0
FOV_WIDE = 65.0

def angular_diff(a, b):
    d = (a - b + 180) % 360 - 180
    return d

def circular_mean(values):
    if not values: return None
    rads = np.deg2rad(values)
    s = np.mean(np.sin(rads)); c = np.mean(np.cos(rads))
    return float(np.rad2deg(np.arctan2(s, c)))

raw_targets = []
for d in raw_detections:
    players = d['players']
    ball = d['ball']
    if not players and not ball:
        raw_targets.append(None)
        continue
    player_lon = circular_mean([p['lon'] for p in players]) if players else None
    player_lat = float(np.mean([p['lat'] for p in players])) if players else None
    if ball and player_lon is not None:
        # weighted mix
        diff = angular_diff(ball['lon'], player_lon)
        tgt_lon = (player_lon + BALL_WEIGHT * diff) % 360
        if tgt_lon > 180: tgt_lon -= 360
        tgt_lat = (1 - BALL_WEIGHT) * player_lat + BALL_WEIGHT * ball['lat']
    elif ball:
        tgt_lon, tgt_lat = ball['lon'], ball['lat']
    else:
        tgt_lon, tgt_lat = player_lon, player_lat
    # FOV: tight if players spread is small, wide if spread is large
    if len(players) > 1:
        lons = np.array([p['lon'] for p in players])
        spread = np.std(lons)
        fov = float(np.clip(FOV_TIGHT + spread * 1.5, FOV_TIGHT, FOV_WIDE))
    else:
        fov = FOV_TIGHT + 10
    raw_targets.append({'t': d['t'], 'lon': tgt_lon, 'lat': tgt_lat, 'fov': fov, 'hasBall': ball is not None})

# Forward-fill gaps (lost detections) with last known
last = None
filled = []
for r in raw_targets:
    if r is None and last is not None:
        filled.append(dict(last))
    else:
        filled.append(r)
        if r is not None: last = r

# EMA smooth (circular for lon)
smoothed = []
sm_lon = sm_lat = sm_fov = None
for r in filled:
    if r is None: continue
    if sm_lon is None:
        sm_lon, sm_lat, sm_fov = r['lon'], r['lat'], r['fov']
    else:
        # Apply dead zone
        d_lon = angular_diff(r['lon'], sm_lon)
        d_lat = r['lat'] - sm_lat
        if abs(d_lon) > DEAD_ZONE_DEG or abs(d_lat) > DEAD_ZONE_DEG * 0.5:
            sm_lon = (sm_lon + EMA_ALPHA * d_lon) % 360
            if sm_lon > 180: sm_lon -= 360
            sm_lat = sm_lat + EMA_ALPHA * d_lat
        sm_fov = sm_fov + EMA_ALPHA * (r['fov'] - sm_fov)
    smoothed.append({'t': r['t'], 'lon': sm_lon, 'lat': sm_lat, 'fov': sm_fov})

# Apply lead (shift future N samples back)
led = []
for i, s in enumerate(smoothed):
    j = min(len(smoothed) - 1, i + LEAD_FRAMES)
    led.append({
        't': s['t'],
        'lon': smoothed[j]['lon'],
        'lat': smoothed[j]['lat'],
        'fov': smoothed[j]['fov'],
    })

print(f'Smoothed track: {len(led)} samples')

Smoothed track: 554 samples


## 6. Write tracks.json (the contract for the client)

In [8]:
import json

tracks = {
    'fps': SAMPLE_HZ,
    'duration': N / FPS,
    'trackingVersion': 1,
    'samples': [[round(s['t'], 3), round(s['lon'], 2), round(s['lat'], 2), round(s['fov'], 1)] for s in led]
}
TRACKS_PATH = os.path.join(OUT_DIR, 'tracks.json')
with open(TRACKS_PATH, 'w') as f:
    json.dump(tracks, f, separators=(',', ':'))
size_kb = os.path.getsize(TRACKS_PATH) / 1024
print(f'Wrote {TRACKS_PATH}: {size_kb:.1f} KB ({len(led)} samples)')
print('Per-minute size:', round(size_kb / (N/FPS/60), 1), 'KB/min')

Wrote /Users/irezaeian/match-tracker/tracking/outputs/tracks.json: 13.5 KB (554 samples)
Per-minute size: 12.9 KB/min


## 7. Render TV-mode preview (THE VERDICT)

Re-projects the equirectangular source through a virtual camera following the smoothed track. Output: `preview_tv.mp4`.

**Watch this video. If it feels watchable → ship it. If not → tune params (EMA_ALPHA, DEAD_ZONE_DEG, LEAD_FRAMES) and re-run from step 5.**

In [9]:
OUT_W, OUT_H = 1280, 720  # 16:9 TV crop

def render_perspective(equi, lon_deg, lat_deg, fov_deg, out_w, out_h):
    """Sample equirectangular `equi` through a perspective camera. Returns BGR image."""
    fh, fw = equi.shape[:2]
    # Build ray grid
    fov = np.deg2rad(fov_deg)
    f = (out_w / 2) / np.tan(fov / 2)
    cx, cy = out_w / 2, out_h / 2
    xs = np.arange(out_w) - cx
    ys = np.arange(out_h) - cy
    xx, yy = np.meshgrid(xs, ys)
    zs = np.full_like(xx, f, dtype=np.float32)
    dirs = np.stack([xx, yy, zs], axis=-1).astype(np.float32)
    norms = np.linalg.norm(dirs, axis=-1, keepdims=True)
    dirs /= norms
    # Rotate by lat (around X), then lon (around Y)
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)
    cl, sl = np.cos(lat), np.sin(lat)
    co, so = np.cos(lon), np.sin(lon)
    Rx = np.array([[1,0,0],[0,cl,-sl],[0,sl,cl]], dtype=np.float32)
    Ry = np.array([[co,0,so],[0,1,0],[-so,0,co]], dtype=np.float32)
    R = Ry @ Rx
    rotated = dirs @ R.T
    x, y, z = rotated[..., 0], rotated[..., 1], rotated[..., 2]
    # Convert to equirectangular UVs
    theta = np.arctan2(x, z)
    phi = np.arcsin(np.clip(y, -1, 1))
    u = (theta / (2 * np.pi) + 0.5) * fw
    v = (phi / np.pi + 0.5) * fh
    return cv2.remap(equi, u.astype(np.float32), v.astype(np.float32),
                     interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)

def lerp_track(t, samples):
    # Binary search
    import bisect
    times = [s[0] for s in samples]
    i = bisect.bisect_left(times, t)
    if i == 0: return samples[0][1], samples[0][2], samples[0][3]
    if i >= len(samples): return samples[-1][1], samples[-1][2], samples[-1][3]
    t0, lon0, lat0, fov0 = samples[i-1]
    t1, lon1, lat1, fov1 = samples[i]
    a = (t - t0) / max(1e-6, t1 - t0)
    # Circular lerp for lon
    dlon = ((lon1 - lon0 + 180) % 360) - 180
    lon = lon0 + a * dlon
    lat = lat0 + a * (lat1 - lat0)
    fov = fov0 + a * (fov1 - fov0)
    return lon, lat, fov

samples = tracks['samples']
TV_PATH = os.path.join(OUT_DIR, 'preview_tv.mp4')
CMP_PATH = os.path.join(OUT_DIR, 'preview_compare.mp4')
cap = cv2.VideoCapture(INPUT_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_tv = cv2.VideoWriter(TV_PATH, fourcc, FPS, (OUT_W, OUT_H))
out_cmp = cv2.VideoWriter(CMP_PATH, fourcc, FPS, (OUT_W, OUT_H * 2))

frame_idx = 0
pbar = tqdm(total=N, desc='Rendering')
while True:
    ret, frame = cap.read()
    if not ret: break
    t = frame_idx / FPS
    lon, lat, fov = lerp_track(t, samples)
    tv = render_perspective(frame, lon, lat, fov, OUT_W, OUT_H)
    out_tv.write(tv)
    # Side-by-side: top = wide equirectangular thumbnail, bottom = TV crop
    wide_thumb = cv2.resize(frame, (OUT_W, OUT_H))
    # Draw indicator of where TV camera is pointing on the wide view
    cx_ind = int((lon + 180) / 360 * OUT_W)
    cy_ind = int((90 - lat) / 180 * OUT_H)
    cv2.rectangle(wide_thumb, (cx_ind-30, cy_ind-20), (cx_ind+30, cy_ind+20), (0, 255, 0), 2)
    stacked = np.vstack([wide_thumb, tv])
    out_cmp.write(stacked)
    frame_idx += 1
    pbar.update(1)
pbar.close()
cap.release()
out_tv.release()
out_cmp.release()
print(f'Done.\n  TV preview:      {TV_PATH}\n  Compare preview: {CMP_PATH}')

Rendering: 100%|██████████| 1880/1880 [01:38<00:00, 19.00it/s]

Done.
  TV preview:      /Users/irezaeian/match-tracker/tracking/outputs/preview_tv.mp4
  Compare preview: /Users/irezaeian/match-tracker/tracking/outputs/preview_compare.mp4


## 7b. 4K vs 5.7K comparison (decide recording resolution)

Re-runs **detection only** (no smoothing, no render) on the same video downscaled to a 4K-equivalent equirectangular (3840×1920). Compares:

- **Ball detection recall** — % of frames where YOLO found the ball at each resolution
- **Position agreement** — average angular error between 4K and 5.7K ball positions on frames where both detected the ball

If 4K recall is within ~5% of 5.7K and angular agreement is <3°, **record at 4K** to save 4× upload time + storage. Otherwise stay at 5.7K.

In [10]:
# Compare ball detection at native 5.7K vs downscaled-to-4K equirectangular.
# Reuses the same equator-band + YOLO pipeline from cell 4, just at two input sizes.

def detect_pass(target_w):
    """Run detection on the video resized to (target_w, target_w//2). Returns per-sample ball lon/lat or None."""
    target_h = target_w // 2
    band_top = int(target_h * 0.33)
    band_bot = int(target_h * 0.67)
    band_h = band_bot - band_top
    cap = cv2.VideoCapture(INPUT_PATH)
    out = []
    fi = 0
    pbar = tqdm(total=N, desc=f'Detect @ {target_w}x{target_h}', leave=False)
    while True:
        ret, frame = cap.read()
        if not ret: break
        if fi % stride == 0:
            if frame.shape[1] != target_w:
                frame_r = cv2.resize(frame, (target_w, target_h))
            else:
                frame_r = frame
            band = frame_r[band_top:band_bot, :, :]
            det_w = 1280
            det_h = int(band_h * det_w / target_w)
            small = cv2.resize(band, (det_w, det_h))
            results = model.predict(small, conf=0.25, classes=[32], verbose=False)
            ball = None
            if len(results) > 0:
                r = results[0]
                for box, cls, conf in zip(r.boxes.xyxy.cpu().numpy(),
                                           r.boxes.cls.cpu().numpy(),
                                           r.boxes.conf.cpu().numpy()):
                    if int(cls) != 32: continue
                    x1, y1, x2, y2 = box
                    cx = (x1 + x2) / 2 * (target_w / det_w)
                    cy_in_band = (y1 + y2) / 2 * (band_h / det_h)
                    cy = band_top + cy_in_band
                    lon = (cx / target_w) * 360.0 - 180.0
                    lat = 90.0 - (cy / target_h) * 180.0
                    if ball is None or conf > ball['conf']:
                        ball = {'lon': lon, 'lat': lat, 'conf': float(conf)}
            out.append(ball)
        fi += 1
        pbar.update(1)
    pbar.close()
    cap.release()
    return out

# Native resolution (whatever the input was — typically 5.7K = 5760x2880)
native_w = W if W % 2 == 0 else W - 1
det_native = detect_pass(native_w)

# 4K-equivalent equirectangular = 3840x1920
det_4k = detect_pass(3840)

n = min(len(det_native), len(det_4k))
hits_native = sum(1 for b in det_native[:n] if b is not None)
hits_4k = sum(1 for b in det_4k[:n] if b is not None)

# Angular error on frames where BOTH detected the ball
errs = []
for a, b in zip(det_native[:n], det_4k[:n]):
    if a is None or b is None: continue
    dlon = ((a['lon'] - b['lon'] + 180) % 360) - 180
    dlat = a['lat'] - b['lat']
    errs.append(float(np.hypot(dlon, dlat)))

print(f'\n=== 4K vs {native_w}x{native_w//2} comparison ===')
print(f'Frames sampled:           {n}')
print(f'Ball recall @ native:     {hits_native}/{n} = {hits_native/n*100:.1f}%')
print(f'Ball recall @ 4K:         {hits_4k}/{n} = {hits_4k/n*100:.1f}%')
print(f'Recall delta (native-4K): {(hits_native - hits_4k)/n*100:+.1f} pp')
if errs:
    print(f'Position agreement:       mean {np.mean(errs):.2f}°, median {np.median(errs):.2f}°, p95 {np.percentile(errs, 95):.2f}° (n={len(errs)})')
else:
    print('Position agreement:       no co-detected frames')

# Verdict
recall_delta = (hits_native - hits_4k) / n * 100
med_err = float(np.median(errs)) if errs else 999
print('\nRECOMMENDATION:')
if recall_delta <= 5 and med_err < 3:
    print('  ✅ Record at 4K — saves ~4× storage/upload, minimal detection loss.')
elif recall_delta <= 10 and med_err < 5:
    print('  ⚠️  Borderline — record 5.7K if upload time is acceptable, else 4K.')
else:
    print('  ❌ Stay at 5.7K — 4K loses too much ball detail at your camera distance.')


=== 4K vs 2560x1280 comparison ===
Frames sampled:           627
Ball recall @ native:     0/627 = 0.0%
Ball recall @ 4K:         0/627 = 0.0%
Recall delta (native-4K): +0.0 pp
Position agreement:       no co-detected frames

RECOMMENDATION:
  ❌ Stay at 5.7K — 4K loses too much ball detail at your camera distance.


## 8. Download outputs

In [11]:
# Outputs are already on disk at OUT_DIR. Open them in Finder / QuickTime:
import subprocess
print(f'Outputs in: {OUT_DIR}')
for f in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, f)
    print(f'  {f:30s}  {os.path.getsize(full)/1024:.1f} KB')

# Uncomment to auto-open in Finder:
# subprocess.run(['open', OUT_DIR])

Outputs in: /Users/irezaeian/match-tracker/tracking/outputs
  preview_compare.mp4             45548.9 KB
  preview_tv.mp4                  15902.2 KB
  tracks.json                     13.5 KB


## What to do with the results

1. Watch `preview_tv.mp4` end-to-end — does it feel like TV?
2. Watch `preview_compare.mp4` — does the green box on the wide view track sensibly?
3. Note any moments where:
   - Camera lags behind action → reduce `EMA_ALPHA` or increase `LEAD_FRAMES`
   - Camera jitters → increase `DEAD_ZONE_DEG`
   - Camera misses a goal or break → likely a detection gap; consider fine-tuning YOLO on your footage
4. Inspect `tracks.json` shape — confirm the schema is what the client needs:
   ```json
   {
     "fps": 10,
     "duration": 60.0,
     "trackingVersion": 1,
     "samples": [[t, lon, lat, fov], ...]
   }
   ```